# Executive Decision Report & Capstone Analytics

## Payment Operations — Descriptive Jupyter Notebook

**Purpose:** Transform the cleaned payment dataset into executive-ready business intelligence findings that can support the final 10–15 slide Executive Decision Report.

### Assignment requirements addressed
- Executive summary and business context
- Analytical methodology
- Revenue and payment-performance drivers
- Conversion funnel analysis
- Risk analysis
- Three strategic initiatives
- ROI projections and implementation roadmap
- Final analytical evidence for the executive presentation

> **Important:** This notebook uses `clean_dataset.csv` produced by the earlier data-cleaning stage. The cleaning logic and the analytical assumptions from the agreed final code are kept intact.

## 1. Executive Question

The analysis is designed around one central business question:

**How can payment operations improve revenue realization, reduce avoidable losses, and generate measurable financial benefit?**

The analysis therefore moves through four levels:

1. **What happened?** — revenue, payment success, refunds, and transaction patterns.
2. **Where did it happen?** — years, months, and payment channels.
3. **Where are the risks/opportunities?** — conversion leakage, COD exposure, refunds, and fee reconciliation.
4. **What could be done?** — UPI incentives, COD risk mitigation, and fee reconciliation, followed by ROI and implementation planning.

## 2. Analytical Methodology

### Data source
The notebook reads the previously cleaned payment dataset from `clean_dataset.csv`.

### Main analytical dimensions
| Dimension | Business purpose |
|---|---|
| Revenue | Measure gross and net monetary performance |
| Payment status | Evaluate transaction success |
| Payment method | Understand channel contribution |
| Time | Identify yearly and monthly patterns |
| Funnel | Measure movement from leads to orders to settlements |
| Refunds | Quantify potential revenue leakage |
| Payment fees | Identify reconciliation opportunity |
| COD | Quantify exposure to high-value COD transactions |
| ROI | Translate initiatives into financial projections |

### Important distinction
The notebook separates **observed dataset metrics** from **simulation/management assumptions**. For example, revenue and refund rates are calculated from the dataset, whereas UPI savings, COD prevention, fee recovery, and the three-year ROI projection are scenario estimates supplied by the assignment/code.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load cleaned dataset
df = pd.read_csv("clean_dataset.csv")

print("Dataset loaded successfully")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

## 3. Dataset Overview

Before interpreting business performance, verify the structure of the cleaned dataset. This prevents conclusions from being drawn from incorrect data types, missing fields, or unexpected categories.

In [ ]:
display(df.head())

print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("Missing values:")
display(df.isna().sum().to_frame("missing_values"))

## 4. Date Preparation

A reliable time dimension is necessary for yearly and monthly trend analysis. The code below detects the first column containing `date` or `time`, converts it to a datetime field, and creates `Year` and `Month` fields.

**Why this matters:** grouping by a proper datetime-derived year/month avoids treating dates as ordinary text and makes the trend analysis reproducible.

In [ ]:
# Detect date/time column
date_col = None
for col in df.columns:
    if "date" in col.lower() or "time" in col.lower():
        date_col = col
        break

print("Detected date column:", date_col)

df["Date"] = pd.to_datetime(df[date_col], errors="coerce")
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

display(df[[date_col, "Date", "Year", "Month"]].head())

## 5. Executive KPI Dashboard

KPIs provide a compact description of the business state. They are not recommendations; they are measurements that executives can use to understand the scale and quality of payment performance.

**Definitions**
- **Gross Revenue:** total `amount_paid`.
- **Net Revenue:** total `net_settlement_amount`.
- **Average Order Value:** mean `amount_paid` per record.
- **Success Rate:** percentage of records whose payment status is `Success`.
- **Total Refunds:** total `refund_amount`.

In [ ]:
kpi_data = {
    "Gross Revenue": [df["amount_paid"].sum()],
    "Net Revenue": [df["net_settlement_amount"].sum()],
    "Avg Order Value": [df["amount_paid"].mean()],
    "Success Rate (%)": [df["payment_status"].eq("Success").mean() * 100],
    "Total Refunds": [df["refund_amount"].sum()]
}

kpis = pd.DataFrame(kpi_data)
display(kpis.style.format({
    "Gross Revenue": "₹{:,.2f}",
    "Net Revenue": "₹{:,.2f}",
    "Avg Order Value": "₹{:,.2f}",
    "Success Rate (%)": "{:.2f}%",
    "Total Refunds": "₹{:,.2f}"
}))

## 6. Yearly Revenue Trend

Yearly aggregation answers whether total transaction value is increasing, decreasing, or fluctuating across the available period. This is a descriptive trend, not a causal explanation.

In [ ]:
yearly = df.groupby("Year")["amount_paid"].sum()
display(yearly.to_frame("Revenue").style.format("₹{:,.2f}"))

plt.figure(figsize=(8, 5))
yearly.plot(kind="bar")
plt.title("Yearly Revenue Trend")
plt.xlabel("Year")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Interpretation framework
When writing the executive report, describe the observed direction and magnitude first. Avoid claiming that a trend was caused by a particular factor unless the dataset or another validated source provides evidence for that causal explanation.

## 7. Monthly Revenue Trend

Monthly analysis identifies seasonality, peaks, troughs, and unusual periods within each year. The assignment's original analysis covers 2023, 2024, and 2025.

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

for year in [2023, 2024, 2025]:
    yearly_data = df[df["Year"] == year].groupby("Month")["amount_paid"].sum()
    print(f"\n{year} monthly revenue:")
    display(yearly_data.to_frame("Revenue").style.format("₹{:,.2f}"))

    plt.figure(figsize=(9, 4))
    yearly_data.plot(marker="o")
    plt.title(f"Monthly Revenue Trend — {year}")
    plt.xlabel("Month")
    plt.ylabel("Revenue (₹)")
    plt.xticks(range(1, 13), month_names)
    plt.grid(True, axis="both", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

## 8. Payment Method Contribution

Payment-method analysis shows how much monetary value is associated with each payment channel. This is useful when considering channel-specific interventions.

The analysis should distinguish **share of revenue** from **number of transactions**. A channel can have a high transaction count but a smaller revenue contribution, or vice versa.

In [ ]:
combined = (
    df.groupby(["Year", "payment_method"])["amount_paid"]
      .sum()
      .unstack()
      .fillna(0)
)

display(combined.style.format("₹{:,.2f}"))

plt.figure(figsize=(9, 5))
combined.plot(kind="bar", stacked=True, ax=plt.gca())
plt.title("Payment Method Comparison (2023–2025)")
plt.xlabel("Year")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Conversion Funnel

The funnel measures movement across three stages:

**Leads → Orders → Successful Settlements**

The two conversion ratios are:

$$Lead\rightarrow Order = \frac{Orders}{Leads}\times100$$

$$Order\rightarrow Settlement = \frac{Successful\ Settlements}{Orders}\times100$$

These ratios help identify where volume is lost. They should be interpreted together with the underlying counts.

In [ ]:
funnel_results = []

for year in [2023, 2024, 2025]:
    df_year = df[df["Year"] == year]
    leads = df_year["lead_id"].nunique() if "lead_id" in df_year.columns else 0
    orders = df_year["order_id"].nunique() if "order_id" in df_year.columns else 0
    settlements = (
        df_year[df_year["payment_status"] == "Success"]["order_id"].nunique()
        if "order_id" in df_year.columns else 0
    )

    lead_order = orders / leads * 100 if leads else np.nan
    order_settlement = settlements / orders * 100 if orders else np.nan

    funnel_results.append({
        "Year": year,
        "Leads": leads,
        "Orders": orders,
        "Settlements": settlements,
        "Lead→Order %": lead_order,
        "Order→Settlement %": order_settlement
    })

funnel_df = pd.DataFrame(funnel_results)
display(funnel_df.style.format({
    "Lead→Order %": "{:.2f}%",
    "Order→Settlement %": "{:.2f}%"
}))

## 10. Refund Risk

Refund rate is calculated as:

$$Refund\ Rate = \frac{Total\ Refunds}{Gross\ Revenue}\times100$$

This provides a simple measure of refund value relative to total sales value. It does not by itself identify why refunds occurred.

In [ ]:
refund_total = df["refund_amount"].sum()
gross_revenue = df["amount_paid"].sum()
refund_rate = refund_total / gross_revenue * 100 if gross_revenue else np.nan

risk_summary = pd.DataFrame({
    "Metric": ["Gross Revenue", "Total Refunds", "Refund Rate %"],
    "Value": [gross_revenue, refund_total, refund_rate]
})

display(risk_summary)

plt.figure(figsize=(6, 4))
plt.bar(["Refund Rate"], [refund_rate])
plt.title("Refund % of Total Sales")
plt.ylabel("Percentage")
plt.annotate(f"{refund_rate:.2f}%", (0, refund_rate), xytext=(0, 5),
             textcoords="offset points", ha="center")
plt.tight_layout()
plt.show()

## 11. Payment Channel Breakdown

For each payment method we calculate:
- total revenue,
- transaction count,
- average transaction value, and
- revenue share.

This creates the evidence base for channel-level strategic initiatives.

In [ ]:
channel = df.groupby("payment_method")["amount_paid"].agg(["sum", "count", "mean"])
channel["share_%"] = channel["sum"] / df["amount_paid"].sum() * 100

channel = channel.rename(columns={
    "sum": "Revenue",
    "count": "Transactions",
    "mean": "Average Transaction Value",
    "share_%": "Revenue Share %"
})

display(channel.style.format({
    "Revenue": "₹{:,.2f}",
    "Average Transaction Value": "₹{:,.2f}",
    "Revenue Share %": "{:.2f}%"
}))

## 12. Strategic Initiative Simulation

The following calculations are **scenario simulations**, not directly observed outcomes.

### Initiative A — UPI Incentive
The original assignment assumes savings equal to **1.5% of UPI transaction volume**.

### Initiative B — COD Risk Mitigation
The original assignment identifies COD transactions above ₹15,000 and assumes **1% of that value** can be protected from logistics waste.

### Initiative C — Fee Reconciliation
The assignment code uses a fixed estimated recovery of **₹4.5 million** from fee reconciliation.

Because these are assumptions, the final presentation should label them clearly as **projected / simulated / estimated**, rather than as realized savings.

In [ ]:
# A. UPI incentive simulation
upi_volume = df[df["payment_method"] == "UPI"]["amount_paid"].sum()
savings_upi = upi_volume * 0.015

# B. COD risk mitigation simulation
cod_high = df[(df["payment_method"] == "COD") & (df["amount_paid"] > 15000)]
prevented_loss = cod_high["amount_paid"].sum() * 0.01

# C. Fee reconciliation assumption from assignment
fee_overcharge = 4.5e6

simulation = pd.DataFrame({
    "Initiative": ["UPI Incentive", "COD Risk Mitigation", "Fee Reconciliation"],
    "Calculated / Assumed Benefit": [savings_upi, prevented_loss, fee_overcharge],
    "Basis": [
        "1.5% × UPI transaction volume",
        "1% × high-value COD transaction value (> ₹15,000)",
        "Fixed assignment estimate"
    ]
})

display(simulation.style.format({"Calculated / Assumed Benefit": "₹{:,.2f}"}))

## 13. Three Strategic Recommendations — Evidence Structure

The capstone requires three strategic recommendations. In the executive deck, each recommendation should follow the same structure:

**Evidence → Business problem → Proposed action → Financial impact → Implementation requirement → Risk/control**

### 1. UPI Incentives
**Evidence to show:** UPI revenue volume and simulated 1.5% benefit.

**Proposed action:** Test targeted incentives intended to shift suitable payment volume toward UPI.

**Measurement:** UPI share, transaction success, incentive cost, incremental net benefit.

### 2. COD Risk Mitigation
**Evidence to show:** high-value COD volume above ₹15,000 and the 1% simulated prevented-loss assumption.

**Proposed action:** Use controls such as OTP verification and/or deposits for defined high-value COD orders.

**Measurement:** COD default rate, return-to-origin/logistics cost, successful delivery rate.

### 3. Fee Reconciliation
**Evidence to show:** fee-reconciliation recovery assumption of ₹4.5 million.

**Proposed action:** Automate payment-gateway fee reconciliation against contractual rates and settlement records.

**Measurement:** disputed fees recovered, reconciliation cycle time, exception rate, unresolved variance.

## 14. ROI Projection

The assignment provides a three-year scenario model. These figures should be presented as **projection assumptions**, not historical performance.

The basic relationship is:

$$Net\ Benefit = Gross\ Margin\ Recovery + Logistics\ Cost\ Savings - Implementation\ Cost$$

The cumulative ROI figures in the supplied code are retained for consistency with the assignment.

In [ ]:
roi = pd.DataFrame({
    "Metric": [
        "Gross Margin Recovery",
        "Logistics Cost Savings",
        "Implementation Cost",
        "Net Benefit",
        "Cumulative ROI %"
    ],
    "Year1": [28500000, 9800000, -6000000, 32300000, 538],
    "Year2": [51100000, 14200000, -2500000, 62800000, 738],
    "Year3": [72400000, 18500000, -2500000, 88400000, 822]
})

display(roi)

### ROI interpretation note
The projection combines modeled benefits and implementation costs supplied by the assignment. Before using it as a real business forecast, management would need to validate the baseline, benefit assumptions, implementation costs, timing, and whether the ROI calculation is based on cumulative or period-specific investment.

## 15. Implementation Roadmap

The supplied assignment code defines a phased rollout:

| Initiative | Start | Full rollout |
|---|---|---|
| UPI Incentives | Q1 2024 | Q2 2024 |
| COD Risk Mitigation | Q2 2024 | Q3 2024 |
| Fee Reconciliation | Q3 2024 | Q4 2024 |

> **Date-quality note:** Because the dataset contains 2023–2025 analysis while this roadmap uses 2024 dates, the final executive report should explicitly identify these as the assignment's planned timeline or update them if the project is being presented as a current implementation plan.

In [ ]:
roadmap = pd.DataFrame({
    "Initiative": ["UPI Incentives", "COD Risk Mitigation", "Fee Reconciliation"],
    "Start": ["Q1 2024", "Q2 2024", "Q3 2024"],
    "Full Rollout": ["Q2 2024", "Q3 2024", "Q4 2024"]
})
display(roadmap)

## 16. Risk & Mitigation Matrix

The capstone requires risk areas and controls to be visible to decision makers.

In [ ]:
risks = pd.DataFrame({
    "Risk": ["High COD default", "Gateway fee overcharge", "UPI adoption lag"],
    "Mitigation": ["OTP + deposit", "Automated reconciliation", "Discount incentive"]
})
display(risks)

## 17. Executive Findings Template

Use the following structure after running all cells and reviewing the actual outputs:

### Finding 1 — Revenue performance
- Observed revenue across the analysis period:
- Highest/lowest year: 
- Key monthly pattern: *

### Finding 2 — Payment-channel structure
- Largest revenue-contributing channel: 
- UPI volume/value: 
- COD high-value exposure: 

### Finding 3 — Conversion and leakage
- Lead→Order conversion: 
- Order→Settlement conversion: 
- Refund rate:

### Finding 4 — Financial opportunity
- UPI simulation: 
- COD mitigation simulation: 
- Fee reconciliation assumption: **₹4.5M**
- Three-year projected net benefit: 

## 18. Mapping the Notebook to the 15-Slide Executive Deck

| Slide | Content | Notebook evidence |
|---:|---|---|
| 1 | Title / business context | Introduction |
| 2 | Executive summary | KPI + findings |
| 3 | Methodology | Methodology section |
| 4 | KPI dashboard | KPI analysis |
| 5 | Revenue trend | Yearly revenue |
| 6 | Monthly/seasonal pattern | Monthly trends |
| 7 | Payment methods | Channel comparison |
| 8 | Conversion funnel | Funnel analysis |
| 9 | Risk / refunds | Refund analysis |
| 10 | Opportunity 1 | UPI simulation |
| 11 | Opportunity 2 | COD simulation |
| 12 | Opportunity 3 | Fee reconciliation |
| 13 | ROI projection | ROI model |
| 14 | Roadmap + risk controls | Roadmap / risk matrix |
| 15 | Executive action plan | Three initiatives + KPIs |
